# Introduction

Let's start with the most obvious **problems** that can be seen by just looking at the database
and here are they:
#### LinkedIn
- [x] the `posted_since` column is relative to the collection date, not absolute
- [x] in the `seniority_level` column there's a value called "Not Applicable"
- [x] Extract the salary using a Q/A extraction model + regex
- [x] Extract the required education using regex
- [x] Extract the skills from the description
#### UpWork
- [x] There are empty strings in the `skills` column
- [x] A lot of columns have useless string components such as *"1978 <ins>hours worked</ins>"*
- [x] Money is represented using strings
- [x] Sometimes values are null and sometimes are strings indicating empty value.
#### Guru
- [x] the `earnings` and `feedback_percent` columns are represented as strings

There are some common problems along every platform I am analyzing here are they<br>
- [x] There are some duplicated IDs of freelancers/jobs
- [x] There is no column indicating when was the data collected which is an important feature
- [x] Data are seperated into multiple tables based on collection date

Some of these problems can be fixed within the inititial query

**NOTE**: In the start of the project I was considering "Data entry" as a data-field job but<br>
because of how far it is from the rest of the data-field jobs I will be execluding it from the<br>
cleaned database.

# Setting up

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
from datetime import datetime, timedelta, date
from typing import Optional, Tuple, List
from IPython.display import display, Markdown
from ipywidgets import IntProgress
import sqlite3
import time
import ast
import sys
import re
import os

RAW_DB_URI = "file:../data/raw_database.db?mode=ro" # XXX
CLEAN_DB_PATH = "../data/clean_database.db"
QA_MODEL_NAME     = "deepset/tinyroberta-squad2"
QA_MODEL_PATH     = "../models/tinyroberta-squad2-model"
QA_TOKENIZER_PATH = "../models/tinyroberta-squad2-tokenizer"

2025-11-13 17:31:47.324083: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-13 17:31:48.483341: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-13 17:31:51.094012: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
with sqlite3.connect(RAW_DB_URI, uri=True) as con:
    # hardcoded i know and don't give a shit

    linkedin_df = pd.read_sql_query("""
        SELECT       *, "2025_09_15" as collection_date  FROM linkedin_jobs_2025_09_15
        UNION SELECT *, "2025_09_25" as collection_date  FROM linkedin_jobs_2025_09_25
        UNION SELECT *, "2025_11_13" as collection_date  FROM linkedin_jobs_2025_11_13
    """, con)
    
    upwork_df = pd.read_sql_query("""
        SELECT       *, "2025_09_15" as collection_date FROM upwork_freelancers_2025_09_15 
        UNION SELECT *, "2025_09_25" as collection_date FROM upwork_freelancers_2025_09_25
        UNION SELECT *, "2025_10_06" as collection_date FROM upwork_freelancers_2025_10_06
        UNION SELECT *, "2025_11_13" as collection_date FROM upwork_freelancers_2025_11_13
    """, con)
    
    guru_df = pd.read_sql_query("""
        SELECT *, "2025_09_15" as collection_date FROM guru_freelancers_2025_09_15
    """, con)

# Data cleaning

### Common problems

Firstly I should remove the duplicates before starting

In [3]:
linkedin_df = linkedin_df.drop_duplicates(subset=["id"], keep="first")
upwork_df   = upwork_df.drop_duplicates(subset=["id"], keep="first")
guru_df     = guru_df.drop_duplicates(subset=["url"], keep="first")

Now let's remove the "Data entry" rules that was collected in the start of the project

In [4]:
linkedin_df = linkedin_df.loc[linkedin_df["searched_job_title"] != "Data entry"]
upwork_df   = upwork_df  .loc[upwork_df["searched_job_title"] != "Data entry"]
guru_df     = guru_df    .loc[guru_df["searched_job_title"] != "Data Entry"]

In [5]:
print("LinkedIn searched job titles: ", linkedin_df["searched_job_title"].unique(), "\n")
print("UpWork searched job titles:", upwork_df["searched_job_title"].unique(), "\n")
print("Guru searched job titles:", guru_df["searched_job_title"].unique(), "\n")

LinkedIn searched job titles:  ['Data engineer' 'Data scientist' 'Machine learning' 'Data analyst'] 

UpWork searched job titles: ['Machine learning' 'Data analyst' 'Data scientist' 'Data engineer'] 

Guru searched job titles: ['Data Engineering' 'Data Science' 'Data Analysis' 'Machine Learning'] 



### LinkedIn

#### Problem 1

Fixing the `posted_since` column to use dates instead of days since the data was collected<br>
NOTE: it will still be an approximatation because linkedin doesn't specify actual posting date

In [6]:
linkedin_df["posted_since"].unique()

array(['1 week ago', '19 hours ago', '1 day ago', '17 hours ago',
       '20 hours ago', '2 days ago', '6 days ago', '3 days ago',
       '6 hours ago', '14 hours ago', '2 weeks ago', '18 hours ago',
       '4 days ago', '11 hours ago', '15 hours ago', '12 hours ago',
       '5 days ago', '13 hours ago', '16 hours ago', '3 weeks ago',
       '1 month ago', '3 hours ago', '5 hours ago', '4 weeks ago',
       '10 hours ago', '22 hours ago', '4 hours ago', '1 hour ago',
       '9 hours ago', '8 hours ago', '2 hours ago', '23 hours ago',
       '21 hours ago', '7 hours ago', '45 minutes ago', '18 minutes ago',
       '29 minutes ago', '56 minutes ago', '53 minutes ago'], dtype=object)

In [7]:
hour_pattern  = re.compile(r"^[0-9]+ hour")
day_pattern   = re.compile(r"^[0-9]+ day")
week_pattern  = re.compile(r"^[0-9]+ week")
month_pattern = re.compile(r"^[0-9]+ month")
year_pattern  = re.compile(r"^[0-9]+ year")

value_pattern = re.compile(r"^[0-9]+")

def parse_posted_since(row: pd.Series) -> date:

    posted_since    = row["posted_since"]
    collection_date = datetime.strptime(row["collection_date"], "%Y_%m_%d")

    if not(collection_date and posted_since):
        return None
    
    used_pattern: re.Pattern = None
    for pattern in [hour_pattern, day_pattern, week_pattern,
                    month_pattern, year_pattern]:
        if re.match(pattern, posted_since):
            used_pattern = pattern
            break

    interval_value = int(value_pattern.search(posted_since).group())
    interval_unit: timedelta = datetime.hour
    
    if used_pattern == hour_pattern:
        interval_unit = timedelta(hours=1)

    elif used_pattern == day_pattern:
        interval_unit = timedelta(days=1)
        
    elif used_pattern == week_pattern:
        interval_unit = timedelta(weeks=1)
        
    elif used_pattern == month_pattern:
        interval_unit = timedelta(days=29.53)
        
    elif  used_pattern == year_pattern:
        interval_unit = timedelta(days=365.25)

    else:
        return np.nan

    interval = interval_value * interval_unit
    
    return datetime.date(collection_date - interval)

In [8]:
linkedin_df["posted_since"] = linkedin_df.apply(
    lambda row: parse_posted_since(row), axis=1)

#### Problem 2

Replacing the "Not Applicable" value in the `seniority_level` column with None

In [9]:
linkedin_df["seniority_level"].unique()

array(['Associate', 'Entry level', 'Mid-Senior level', 'Not Applicable',
       'Internship', 'Director', 'Executive'], dtype=object)

In [10]:
linkedin_df["seniority_level"] = linkedin_df["seniority_level"].replace(
    "Not Applicable", None)

#### Problem 3

Now let's try to extract the salary using an extractive Q/A model + regex

In [11]:
if os.path.isfile(QA_MODEL_PATH) and \
   os.path.isfile(QA_TOKENIZER_PATH):
    model     = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_PATH)
    tokenizer = AutoTokenizer.from_pretrained(QA_TOKENIZER_PATH)
else:
    model     = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
    model    .save_pretrained(QA_MODEL_PATH)
    tokenizer.save_pretrained(QA_TOKENIZER_PATH)

qa_model = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer
)

Device set to use cpu


In [12]:
def extract_salary_range(desc: str) -> Optional[Tuple]:
    """
    params: 
        desc: the job description as a string
    returns: an tuple of the salary range but it would return None if it wansn't found
    This function aims to eliminate "false positives", it's ok to have some true negatives
    
    NOTE: this function doesn't work well with job description from any language but 
    English
    """
    if not(isinstance(desc, str)):
        return None

    question = "What is the salary range for the role?"
    output = qa_model(question = question, context = desc)

    if output["score"] < 0.3: 
        return None

    matches = re.findall(
        r"[\$|€|-| ]([0-9,.k]+)",
        output["answer"],
        flags=re.IGNORECASE
    )
    # print(matches)
    
    if not(matches):
        return None
    if len(matches) > 2:
        matches = matches[:2]
    elif len(matches) == 1:
        matches = [matches[0], matches[0]]
    
    salary_range: list = [None, None] # will be converted into a tuple
    
    for i in range(2):
        is_hour_rate: bool = False
        mag: float = 1.0

        if "," in matches[i]:
            if len(matches[i].split(",")[1]) == 2:
                matches[i] = matches[i].replace(",", ".")
                is_hour_rate = True

        if "k" in matches[i].lower():
            mag = 1000.0

        matches[i] = matches[i].lower()
        matches[i] = matches[i].replace(",", "")
        matches[i] = matches[i].replace("k", "")

        salary_range[i] = float(matches[i]) * mag

    return tuple(salary_range)

In [13]:
progress_bar = IntProgress(min=0, max=len(linkedin_df["description"]))
display(Markdown("Extracting salaries from job descriptions"))
display(progress_bar)

def extract_and_update_progress(description: str):
    progress_bar.value += 1
    return extract_salary_range(description)

salary_ranges = linkedin_df["description"].apply(extract_and_update_progress)

linkedin_df["salary_min"] = salary_ranges.apply(lambda x: list(x)[0] if x is not None else x)
linkedin_df["salary_max"] = salary_ranges.apply(lambda x: list(x)[1] if x is not None else x)

Extracting salaries from job descriptions

IntProgress(value=0, max=2578)

Now I need to seperate hour pays from annaul salaries

In [14]:
linkedin_df["pay_type"] = linkedin_df["salary_min"].apply(
    lambda s: None if np.isnan(s) else "Hourly" if s < 200 else "Annually"
)

#### Problem 4

Extracting skills from the job descriptions, the skills I am searching for are the ones<br>
those exist in the UpWork skills column, instead of writing every skill I am searching for<br>
manualy.<br>

In [15]:
skills = [
    # Programming & Scripting
    "python", " r ", "java", "scala ", "c++", "c#", " go ", "bash", "sql", "nosql",
    "vba", "powershell", "scala,", " r,", " go,",
    
    # Data Handling & Databases
    "mysql", "postgresql", "sqlite", "oracle", "mssql", "mongodb", "cassandra",
    "dynamodb", "redis", "elasticsearch", "neo4j", "snowflake", "bigquery", 
    "redshift", "cosmosdb", "athena"
    
    # Data Processing & ETL
    "pandas", "numpy", "dask", "polars", "pyarrow", "koalas", 
    "airflow", "luigi", "prefect", " dbt", 
    "spark", "pyspark", "hive", "pig", "beam", "flink", "kafka",
    
    # Visualization & BI
    "matplotlib", "seaborn", "plotly", "bokeh", "altair",
    "powerbi", "tableau", "looker", "qlik",
    
    # Cloud & DevOps
    "aws", " gcp", "azure", "databricks", " emr", "sagemaker",
    "docker", "kubernetes", "terraform", "jenkins", "git", "github", "gitlab",
    "ci/cd",
    
    # Machine Learning
    "scikit learn", "scikit-learn", "xgboost", "lightgbm", "catboost", "mlflow",
    "pytorch", "tensorflow", "keras", "jax", "fastai", "onnx", 
    "opencv", "nltk", "spacy", "transformers", "huggingface",
    
    # Statistics & Math
    "statistics", "probability", "linear algebra", "calculus", "optimization",
    
    # Big Data & Distributed Systems
    "hadoop", "hdfs", "yarn", "mapreduce", "zookeeper", 
    
    # Data Engineering & Streaming
    " etl ", " elt ", "data pipeline", "streaming", "batch processing",
    
    # MLOps & Deployment
    "mlops", "model serving", "feature store", "kubeflow", "tfx", " ray", "nltk",
    "nlp",
    
    # General Skills
    "excel", "vba", "regex", "json", "xml", "yaml", "parquet", " orc", "avro",
    "api", " rest ", "grpc", "graphql",
    
    # Soft Skills / Business
    "communication", "storytelling", "problem solving",
    "teamwork", "critical thinking", "project management", "agile", "scrum"
]

def extract_skills(desc: str) -> List[str]:
    found_skills = []
    for skill in skills:
        if skill.lower() in desc.lower():
            found_skills.append(skill)
            
    return found_skills

In [16]:
linkedin_df["skills"] = linkedin_df["description"].apply(
    lambda s: ",".join(extract_skills(s))
)

In [17]:
linkedin_df[["skills", "salary_min", "pay_type"]].sample(7)

,skills,salary_min,pay_type
2874,"sql,tableau,qlik,git,statistics,excel,api,comm...",NaN,None
2421,"python,sql, r,,git,statistics,excel",141000.0,Annually
1479,"sql,oracle,hive,excel,communication",NaN,None
2255,"python,sql,databricks,data pipeline",50000.0,Annually
2044,"excel, orc",NaN,None
35,"python,bash,spark,flink,kafka,aws,docker,kuber...",NaN,None
2482,"python,sql, r,,snowflake, dbt,scikit-learn,sta...",NaN,None


#### Problem 5

Now let's do the same as we did at [Problem 3](#problem-3) and extract the required education from<br>
the job descriptions using the same model

In [18]:
def extract_required_education(desc: str) -> str:
    """
    params:
        takes the description of a job as a string
    returns:
        returns one of the following ["Bachelor", "Master's", "PhD", "High School", "Not mentioned"]
        based on regex matching
    """
    if not(isinstance(desc, str)):
        return "Not mentioned"

    if re.search(r"high school", desc, re.IGNORECASE):
       return "High School" 

    elif re.search(r"bachelor| b\.s | bs\.", desc, re.IGNORECASE): # bs for bachelor not bullshit (nvm they are the same thing)
        return "Bachelor"

    elif re.search(r"master\'s|masters", desc, re.IGNORECASE):
        return "Master's"

    elif re.search(r"phd|ph\.d", desc, re.IGNORECASE):
        return "PhD"

    else:
        return "Not mentioned"

In [19]:
linkedin_df["education"] = linkedin_df["description"].apply(extract_required_education)

### UpWork

#### Problem 1

Fixing the empty strings in the `skills` column

In [20]:
list(upwork_df["skills"].sample(1))

["['Data Analytics & Visualization Software', 'Dashboard', 'Data Analysis', 'Microsoft Excel', 'Google Sheets', '', '', '', '', '', '', '', '', '', '']"]

In [21]:
upwork_df["skills"] = upwork_df["skills"].apply(
    lambda list_: ", ".join((list(filter(lambda s: len(s) > 0, ast.literal_eval(list_)))))
)

In [22]:
upwork_df["skills"].sample(1).iloc[0]

'SQL, Microsoft Excel, Business Intelligence, Data Analysis, Google Analytics, Microsoft Power BI, R'

#### Problem 2

Now let's remove the clutter strings from some of the columns

In [23]:
upwork_df[["hours_worked", "hourly_jobs_done", "fixed_jobs_done"]].sample(5)

,hours_worked,hourly_jobs_done,fixed_jobs_done
604,20163 hours worked,16 hourly jobs,4 fixed price jobs
715,5436 hours worked,13 hourly jobs,4 fixed price jobs
514,52 hours worked,5 hourly jobs,6 fixed price jobs
1126,None,None,None
1655,3313 hours worked,37 hourly jobs,17 fixed price jobs


In [24]:
def extract_value(s: str) -> int | None:
    value_pattern = re.compile("[0-9]+")

    if not(isinstance(s, str)):
        return None

    match = value_pattern.search(s)

    if not(match):
        return None

    return int(match.group())

for col in ["hours_worked", "hourly_jobs_done", "fixed_jobs_done"]:
    upwork_df[col] = upwork_df[col].apply(extract_value)

In [25]:
upwork_df["hours_worked"].sample(5)

27      4521.0
1178     537.0
1940    2810.0
688      676.0
1099       NaN
Name: hours_worked, dtype: float64

#### Problem 3

Converting the money format from being a string into being a float for the `hour_rate` and `earnings`<br>
columns

In [26]:
upwork_df[["earnings", "hour_rate"]].head(5)

,earnings,hour_rate
0,$9K+ earned,$20
1,$2K+ earned,$25
2,$300K+ earned,$95
3,$400+ earned,$35
4,None,$20


In [27]:
hour_rate_pattern = re.compile(r"\$[0-9.]+")
earnings_pattern = re.compile(r"\$[0-9]+")

def extract_earnings(s: str) -> int | None:
    if not(isinstance(s, str)):
        return None

    match = earnings_pattern.search(s)
    if not(match):
        return None

    magnitude = 1
    if "K" in s:
        magnitude = 1000
    elif "M" in s:
        magnitude = 1000_000

    return int(match.group()[1:]) * magnitude

def extract_hour_rate(s: str) -> float | None:
    if not(isinstance(s, str)):
        return None

    match = hour_rate_pattern.search(s)

    if not(match):
        return None

    return float(match.group()[1:])

upwork_df["earnings"] = upwork_df["earnings"].apply(extract_earnings)
upwork_df["hour_rate"] = upwork_df["hour_rate"].apply(extract_hour_rate)

In [28]:
upwork_df[["earnings", "hour_rate"]].head(5)

,earnings,hour_rate
0,9000.0,20.0
1,2000.0,25.0
2,300000.0,95.0
3,400.0,35.0
4,NaN,20.0


### Guru

#### Problem 1

Let's fix the `feedback_percent` and `earnings` format & dtype

In [29]:
guru_df[["feedback_percent", "earnings"]].head(5)

,feedback_percent,earnings
200,None,$0
201,None,$0
202,None,$0
203,None,$0
204,None,$0


In [30]:
def extract_feedback(s: str) -> float | None:
    pattern = re.compile(r"[0-9.,]+")

    if not(isinstance(s, str)):
        return None

    match = pattern.search(s)

    if not(match):
        return None

    return float(match.group())

def extract_earnings(s: str) -> int | None:
    pattern = re.compile(r"\$[0-9,,]+")

    if not(isinstance(s, str)):
        return None

    match = pattern.search(s)

    if not(match):
        return None

    magnitude = 1
    if "K" in s:
        magnitude = 1000
    elif "M" in s:
        magnitude = 1000_000
    
    earnings_str = match.group()[1:]
    earnings_str = earnings_str.replace(",", "")

    return float(earnings_str) * magnitude

guru_df["feedback_percent"] = guru_df["feedback_percent"].apply(extract_feedback)
guru_df["earnings"] = guru_df["earnings"].apply(extract_earnings)

In [31]:
guru_df[["feedback_percent", "earnings"]].head(5)

,feedback_percent,earnings
200,NaN,0.0
201,NaN,0.0
202,NaN,0.0
203,NaN,0.0
204,NaN,0.0


# Data storing

In [32]:
with sqlite3.connect(CLEAN_DB_PATH) as con:
    linkedin_df.to_sql("linkedin", con, if_exists="replace", index=False)
    upwork_df.to_sql("upwork", con, if_exists="replace", index=False)
    guru_df.to_sql("guru", con, if_exists="replace", index=False)